# Incident Response Runbook: Credential Access - Brute Force

**Tactic:** Credential Access
**Technique:** Brute Force (T1110)
**Severity:** HIGH

## Overview

This runbook provides step-by-step incident response procedures for Brute Force activities.

## Incident Response Phases

1. **Detection & Analysis**
2. **Containment**
3. **Eradication**
4. **Recovery**
5. **Post-Incident Activities**


## Phase 1: Detection & Analysis

### Objectives
- Confirm the incident
- Identify the scope
- Assess impact


In [ ]:
import json
import re
from datetime import datetime
import subprocess
import os

# Import security integrations
from autobook.runbook_loader import *
from splunk.splunk_data_collector import SplunkDataCollector
from crowdstrike.crowdstrike_response import CrowdStrikeResponse
from iris.iris_integration import IRISIntegration
from misp.misp_integration import MISPIntegration
from shuffle.shuffle_integration import ShuffleIntegration

# Initialize integrations
splunk = SplunkDataCollector()
crowdstrike = CrowdStrikeResponse()
iris = IRISIntegration()
misp = MISPIntegration()
shuffle = ShuffleIntegration()

# Step 1: Detection & Analysis
print("=" * 60)
print("STEP 1: Detection & Analysis")
print("=" * 60)

alert_data = {
    'alert_id': 'ALERT-001',
    'timestamp': datetime.now().isoformat(),
    'tactic': 'Credential Access',
    'technique': 'Brute Force',
    'severity': 'HIGH',
    'incident_id': f"IR-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
}

print(f"Alert ID: {alert_data['alert_id']}")
print(f"Timestamp: {alert_data['timestamp']}")
print(f"Tactic: {alert_data['tactic']}")
print(f"Technique: {alert_data['technique']}")
print(f"Severity: {alert_data['severity']}")
print(f"Incident ID: {alert_data['incident_id']}")

# Query Splunk for brute force indicators
print(f"\n[ACTION] Querying Splunk for brute force indicators...")
brute_force_queries = [
    # Failed authentication attempts
    'index=* (sourcetype=WinEventLog:Security OR sourcetype=linux_secure) EventCode=4625 OR "authentication failure" OR "invalid password" | stats count by src_ip,user | where count > 5',
    # RDP brute force
    'index=* sourcetype=WinEventLog:Security EventCode=4625 source="RDP" | stats count by src_ip | where count > 10',
    # SSH brute force
    'index=* sourcetype=linux_secure "Failed password" | stats count by src_ip,user | where count > 3',
    # VPN brute force
    'index=* sourcetype=vpn_logs "authentication failed" OR "login failed" | stats count by src_ip | where count > 5',
    # Web application brute force
    'index=* sourcetype=access_combined (status=401 OR "invalid credentials") | stats count by clientip | where count > 10'
]

suspicious_activities = []
affected_systems = []

for query in brute_force_queries:
    try:
        results = splunk.search(query, time_range='-24h')
        if results:
            suspicious_activities.extend(results)
            print(f"   Found {len(results)} brute force indicators")
        else:
            print(f"  - No results for query: {query[:50]}...")
    except Exception as e:
        print(f"   Query failed: {e}")

# Analyze with CrowdStrike for brute force patterns
print(f"\n[ACTION] Analyzing with CrowdStrike for brute force patterns...")
for activity in suspicious_activities:
    try:
        host_analysis = crowdstrike.analyze_host(activity.get('host', ''))
        if host_analysis.get('brute_force_detected'):
            affected_systems.append({
                'hostname': activity.get('host', ''),
                'device_id': host_analysis.get('device_id', ''),
                'brute_force_indicators': host_analysis.get('indicators', []),
                'attack_vectors': host_analysis.get('attack_vectors', [])
            })
            print(f"   Brute force detected on {activity.get('host', '')}: {len(host_analysis.get('indicators', []))} indicators, {len(host_analysis.get('attack_vectors', []))} attack vectors")
    except Exception as e:
        print(f"   Host analysis failed for {activity.get('host', '')}: {e}")

# Enrich with threat intelligence
print(f"\n[ACTION] Enriching with threat intelligence...")
for activity in suspicious_activities:
    try:
        # Search for attacking IPs in MISP
        if activity.get('src_ip'):
            misp_results = misp.search_iocs(activity['src_ip'], type='ip')
            if misp_results:
                activity['threat_intel'] = misp_results[:3]
                print(f"   Threat intel found for IP {activity.get('src_ip')}: {len(misp_results)} matches")
    except Exception as e:
        print(f"   Threat intel lookup failed: {e}")

# Create IRIS case
print(f"\n[ACTION] Creating IRIS incident case...")
try:
    iris_case = iris.create_case(
        title=f"Credential Access - Brute Force Incident {alert_data['incident_id']}",
        description=f"Detected brute force attacks on {len(affected_systems)} systems with {len(suspicious_activities)} suspicious indicators.",
        severity='high',
        tags=['credential-access', 'brute-force', 't1110']
    )
    alert_data['iris_case_id'] = iris_case.get('case_id')
    print(f"   IRIS case created: {iris_case.get('case_id')}")
except Exception as e:
    print(f"   IRIS case creation failed: {e}")

print(f"\n Detection complete:")
print(f"  - {len(suspicious_activities)} suspicious brute force activities detected")
print(f"  - {len(affected_systems)} affected systems identified")
print(f"  - Threat intelligence enriched: {len([a for a in suspicious_activities if a.get('threat_intel')])} activities")
print(f"  - IRIS case created: {alert_data.get('iris_case_id', 'none')}")


## Phase 2: Containment

### Objectives
- Stop the attack
- Isolate affected systems
- Prevent spread


In [ ]:
# Step 2: Containment
print("\n" + "=" * 60)
print("STEP 2: Containment")
print("=" * 60)

isolated_hosts = []
blocked_ips = []
rate_limited_services = []

# Block attacking IPs immediately
print(f"\n[ACTION] Blocking attacking IPs...")
for activity in suspicious_activities:
    try:
        if activity.get('src_ip'):
            block_result = shuffle.block_ip(activity['src_ip'])
            if block_result.get('status') == 'blocked':
                blocked_ips.append(activity['src_ip'])
                print(f"   Blocked attacking IP: {activity['src_ip']}")
    except Exception as e:
        print(f"   IP blocking failed for {activity.get('src_ip', 'unknown')}: {e}")

# Isolate systems under heavy attack
print(f"\n[ACTION] Isolating systems under heavy attack...")
for system in affected_systems:
    try:
        # Check if system is under sustained attack
        attack_intensity = len(system.get('brute_force_indicators', []))
        if attack_intensity > 50:  # Threshold for isolation
            isolation_result = crowdstrike.isolate_host(system['device_id'])
            if isolation_result.get('status') == 'isolated':
                isolated_hosts.append(system['hostname'])
                print(f"   Isolated heavily attacked system: {system['hostname']} ({attack_intensity} indicators)")
    except Exception as e:
        print(f"   Isolation failed for {system.get('hostname', 'unknown')}: {e}")

# Implement rate limiting on authentication services
print(f"\n[ACTION] Implementing rate limiting on authentication services...")
for system in affected_systems:
    try:
        for vector in system.get('attack_vectors', []):
            if 'rdp' in vector.lower():
                rdp_limit = shuffle.rate_limit_service(system['hostname'], 'rdp', max_attempts=3)
                if rdp_limit.get('status') == 'limited':
                    rate_limited_services.append(f"{system['hostname']}:RDP")
                    print(f"   Rate limited RDP on {system['hostname']}")
            if 'ssh' in vector.lower():
                ssh_limit = shuffle.rate_limit_service(system['hostname'], 'ssh', max_attempts=3)
                if ssh_limit.get('status') == 'limited':
                    rate_limited_services.append(f"{system['hostname']}:SSH")
                    print(f"   Rate limited SSH on {system['hostname']}")
            if 'vpn' in vector.lower():
                vpn_limit = shuffle.rate_limit_service(system['hostname'], 'vpn', max_attempts=3)
                if vpn_limit.get('status') == 'limited':
                    rate_limited_services.append(f"{system['hostname']}:VPN")
                    print(f"   Rate limited VPN on {system['hostname']}")
    except Exception as e:
        print(f"   Rate limiting failed for {system.get('hostname', 'unknown')}: {e}")

# Enable account lockout policies
print(f"\n[ACTION] Enabling account lockout policies...")
lockout_policies = []
try:
    lockout_result = shuffle.enable_account_lockout(
        threshold=5,  # Lock after 5 failed attempts
        duration=900  # Lock for 15 minutes
    )
    if lockout_result.get('status') == 'enabled':
        lockout_policies.append('account_lockout')
        print(f"   Enabled account lockout policy (5 attempts, 15min lock)")
except Exception as e:
    print(f"   Account lockout policy failed: {e}")

# Terminate suspicious authentication processes
print(f"\n[ACTION] Terminating suspicious authentication processes...")
terminated_processes = []
for activity in suspicious_activities:
    try:
        if activity.get('process_id') and activity.get('host'):
            termination_result = crowdstrike.terminate_process(
                activity['host'],
                activity['process_id']
            )
            if termination_result.get('status') == 'terminated':
                terminated_processes.append(f"{activity['host']}:{activity['process_id']}")
                print(f"   Terminated suspicious authentication process {activity['process_id']} on {activity['host']}")
    except Exception as e:
        print(f"   Process termination failed: {e}")

# Enable enhanced monitoring for brute force
print(f"\n[ACTION] Enabling enhanced monitoring for brute force...")
try:
    monitoring_result = shuffle.enable_enhanced_monitoring(
        targets=[s['hostname'] for s in affected_systems],
        rules=['brute_force_detection', 'failed_authentication_spikes', 'suspicious_login_patterns']
    )
    if monitoring_result.get('status') == 'enabled':
        print(f"   Enhanced brute force monitoring enabled for {len(affected_systems)} systems")
except Exception as e:
    print(f"   Enhanced monitoring setup failed: {e}")

print(f"\n Containment complete:")
print(f"  - {len(blocked_ips)} attacking IPs blocked")
print(f"  - {len(isolated_hosts)} heavily attacked systems isolated")
print(f"  - {len(rate_limited_services)} services rate limited")
print(f"  - {len(lockout_policies)} account lockout policies enabled")
print(f"  - {len(terminated_processes)} suspicious processes terminated")


## Phase 3: Eradication

### Objectives
- Remove malicious artifacts
- Close vulnerabilities
- Verify threat removal


In [ ]:
# Step 3: Eradication
print("\n" + "=" * 60)
print("STEP 3: Eradication")
print("=" * 60)

cleaned_logs = []
blocked_attackers = []
hardened_services = []

# Clean authentication logs of brute force attempts
print(f"\n[ACTION] Cleaning authentication logs...")
for system in affected_systems:
    try:
        log_cleanup = crowdstrike.clean_authentication_logs(system['device_id'])
        if log_cleanup.get('status') == 'cleaned':
            cleaned_logs.append(system['hostname'])
            print(f"   Cleaned authentication logs on {system['hostname']}")
    except Exception as e:
        print(f"   Log cleanup failed for {system.get('hostname', 'unknown')}: {e}")

# Block all identified attacker IPs permanently
print(f"\n[ACTION] Blocking all identified attacker IPs permanently...")
for activity in suspicious_activities:
    try:
        if activity.get('src_ip') and activity.get('src_ip') not in blocked_ips:
            permanent_block = shuffle.block_ip_permanently(activity['src_ip'])
            if permanent_block.get('status') == 'blocked':
                blocked_attackers.append(activity['src_ip'])
                print(f"   Permanently blocked attacker IP: {activity['src_ip']}")
    except Exception as e:
        print(f"   Permanent IP blocking failed for {activity.get('src_ip', 'unknown')}: {e}")

# Harden authentication services
print(f"\n[ACTION] Hardening authentication services...")
for system in affected_systems:
    try:
        for vector in system.get('attack_vectors', []):
            if 'rdp' in vector.lower():
                rdp_hardening = crowdstrike.harden_service(system['device_id'], 'rdp')
                if rdp_hardening.get('status') == 'hardened':
                    hardened_services.append(f"{system['hostname']}:RDP")
                    print(f"   Hardened RDP service on {system['hostname']}")
            if 'ssh' in vector.lower():
                ssh_hardening = crowdstrike.harden_service(system['device_id'], 'ssh')
                if ssh_hardening.get('status') == 'hardened':
                    hardened_services.append(f"{system['hostname']}:SSH")
                    print(f"   Hardened SSH service on {system['hostname']}")
    except Exception as e:
        print(f"   Service hardening failed for {system.get('hostname', 'unknown')}: {e}")

# Reset compromised accounts
print(f"\n[ACTION] Resetting potentially compromised accounts...")
reset_accounts = []
for system in affected_systems:
    try:
        # Reset accounts that had successful logins during brute force period
        account_reset = shuffle.reset_compromised_accounts(system['hostname'])
        if account_reset.get('status') == 'reset':
            reset_accounts.extend(account_reset.get('accounts_reset', []))
            print(f"   Reset {len(account_reset.get('accounts_reset', []))} potentially compromised accounts on {system['hostname']}")
    except Exception as e:
        print(f"   Account reset failed for {system.get('hostname', 'unknown')}: {e}")

# Remove brute force tools and scripts
print(f"\n[ACTION] Removing brute force tools and scripts...")
removed_tools = []
for activity in suspicious_activities:
    try:
        if activity.get('tool_path') and activity.get('host'):
            tool_removal = crowdstrike.remove_file(
                activity['host'],
                activity['tool_path']
            )
            if tool_removal.get('status') == 'removed':
                removed_tools.append(activity['tool_path'])
                print(f"   Removed brute force tool: {activity['tool_path']}")
    except Exception as e:
        print(f"   Tool removal failed for {activity.get('tool_path', 'unknown')}: {e}")

# Implement geo-blocking for suspicious regions
print(f"\n[ACTION] Implementing geo-blocking for suspicious regions...")
geo_blocks = []
try:
    # Block IPs from high-risk countries associated with brute force
    suspicious_countries = ['CN', 'RU', 'BR', 'IN']  # Example high-risk countries
    for country in suspicious_countries:
        geo_block = shuffle.block_country(country, services=['rdp', 'ssh', 'vpn'])
        if geo_block.get('status') == 'blocked':
            geo_blocks.append(country)
            print(f"   Blocked access from {country} for authentication services")
except Exception as e:
    print(f"   Geo-blocking failed: {e}")

# Verify threat removal
print(f"\n[ACTION] Verifying threat removal...")
verification_results = []
for system in affected_systems:
    try:
        verify_result = crowdstrike.verify_brute_force_removal(system['device_id'])
        verification_results.append({
            'system': system['hostname'],
            'status': 'clean' if verify_result.get('brute_force_detected', True) == False else 'threats_remaining',
            'remaining_indicators': verify_result.get('remaining_indicators', 0)
        })
        status = "" if verify_result.get('brute_force_detected', True) == False else ""
        print(f"  {status} Verification for {system['hostname']}: {verify_result.get('remaining_indicators', 0)} brute force indicators remaining")
    except Exception as e:
        print(f"   Verification failed for {system.get('hostname', 'unknown')}: {e}")

print(f"\n Eradication complete:")
print(f"  - {len(cleaned_logs)} systems had authentication logs cleaned")
print(f"  - {len(blocked_attackers)} additional attacker IPs permanently blocked")
print(f"  - {len(hardened_services)} authentication services hardened")
print(f"  - {len(reset_accounts)} potentially compromised accounts reset")
print(f"  - {len(removed_tools)} brute force tools removed")
print(f"  - {len(geo_blocks)} countries blocked for authentication services")
print(f"  - {len([v for v in verification_results if v['status'] == 'clean'])} systems verified clean")


## Phase 4: Recovery

### Objectives
- Restore systems
- Validate functionality
- Monitor for issues


In [ ]:
# Step 4: Recovery
print("\n" + "=" * 60)
print("STEP 4: Recovery")
print("=" * 60)

restored_services = []
reconnected_systems = []
validated_authentication = []

# Restore authentication services with enhanced security
print(f"\n[ACTION] Restoring authentication services with enhanced security...")
for system in affected_systems:
    try:
        if system['hostname'] in isolated_hosts:
            service_restore = crowdstrike.restore_authentication_services(system['device_id'])
            if service_restore.get('status') == 'restored':
                restored_services.append(system['hostname'])
                print(f"   Restored authentication services on {system['hostname']}")
    except Exception as e:
        print(f"   Service restoration failed for {system.get('hostname', 'unknown')}: {e}")

# Reconnect isolated systems
print(f"\n[ACTION] Reconnecting isolated systems...")
for system in affected_systems:
    try:
        if system['hostname'] in isolated_hosts:
            reconnect_result = crowdstrike.reconnect_host(system['device_id'])
            if reconnect_result.get('status') == 'reconnected':
                reconnected_systems.append(system['hostname'])
                print(f"   Reconnected {system['hostname']} to network")
    except Exception as e:
        print(f"   Reconnection failed for {system.get('hostname', 'unknown')}: {e}")

# Validate authentication system integrity
print(f"\n[ACTION] Validating authentication system integrity...")
for system in affected_systems:
    try:
        auth_validation = crowdstrike.validate_authentication_integrity(system['device_id'])
        if auth_validation.get('status') == 'validated':
            validated_authentication.append(system['hostname'])
            print(f"   Authentication integrity validated for {system['hostname']}")
        else:
            issues = auth_validation.get('issues', [])
            print(f"   Authentication issues found for {system['hostname']}: {len(issues)} issues")
    except Exception as e:
        print(f"   Authentication validation failed for {system.get('hostname', 'unknown')}: {e}")

# Implement advanced authentication protections
print(f"\n[ACTION] Implementing advanced authentication protections...")
advanced_protections = []
try:
    # Deploy multi-factor authentication
    mfa_result = shuffle.enforce_mfa_policy(
        accounts='all',
        required=True,
        exceptions=['service_accounts']
    )
    if mfa_result.get('status') == 'enforced':
        advanced_protections.append('mfa_policy')
        print(f"   Enforced MFA policy for all user accounts")

    # Implement adaptive authentication
    adaptive_result = shuffle.enable_adaptive_authentication(
        risk_based=True,
        location_based=True,
        device_based=True
    )
    if adaptive_result.get('status') == 'enabled':
        advanced_protections.append('adaptive_auth')
        print(f"   Enabled adaptive authentication")
except Exception as e:
    print(f"   Advanced protection implementation failed: {e}")

# Set up continuous brute force monitoring
print(f"\n[ACTION] Setting up continuous brute force monitoring...")
monitoring_systems = []
for system in affected_systems:
    try:
        brute_force_monitoring = crowdstrike.setup_brute_force_monitoring(
            system['device_id'],
            rules=['failed_login_spikes', 'geographic_anomalies', 'time_based_anomalies']
        )
        if brute_force_monitoring.get('status') == 'monitoring':
            monitoring_systems.append(system['hostname'])
            print(f"   Continuous brute force monitoring enabled for {system['hostname']}")
    except Exception as e:
        print(f"   Brute force monitoring setup failed for {system.get('hostname', 'unknown')}: {e}")

# Update security policies
print(f"\n[ACTION] Updating security policies to prevent brute force...")
policy_updates = []
try:
    # Enhance authentication monitoring
    auth_policy = crowdstrike.update_security_policy(
        policy_type='authentication_monitoring',
        rules={
            'failed_login_alerting': True,
            'geo_velocity_checks': True,
            'account_lockout_enhanced': True,
            'suspicious_ip_detection': True
        }
    )
    if auth_policy.get('status') == 'updated':
        policy_updates.append('authentication_monitoring')
        print(f"   Updated authentication monitoring policy")

    # Update network access controls
    network_policy = shuffle.update_security_policy(
        policy_type='network_access',
        rules={
            'ip_whitelisting': True,
            'geo_blocking': True,
            'vpn_enforcement': True,
            'zero_trust_networking': True
        }
    )
    if network_policy.get('status') == 'updated':
        policy_updates.append('network_access')
        print(f"   Updated network access control policy")
except Exception as e:
    print(f"   Policy update failed: {e}")

# Implement fail2ban-like automated blocking
print(f"\n[ACTION] Implementing automated IP blocking...")
try:
    auto_block_result = shuffle.enable_automated_blocking(
        trigger='failed_logins',
        threshold=10,
        duration=3600,  # 1 hour block
        services=['rdp', 'ssh', 'vpn', 'web']
    )
    if auto_block_result.get('status') == 'enabled':
        policy_updates.append('automated_blocking')
        print(f"   Enabled automated IP blocking for failed authentication attempts")
except Exception as e:
    print(f"   Automated blocking setup failed: {e}")

print(f"\n Recovery complete:")
print(f"  - {len(restored_services)} systems had authentication services restored")
print(f"  - {len(reconnected_systems)} isolated systems reconnected")
print(f"  - {len(validated_authentication)} systems had authentication validated")
print(f"  - {len(advanced_protections)} advanced authentication protections implemented")
print(f"  - {len(monitoring_systems)} systems under continuous brute force monitoring")
print(f"  - {len(policy_updates)} security policies updated")


## Phase 5: Post-Incident Activities

### Objectives
- Document findings
- Implement improvements
- Share lessons learned


In [ ]:
# Step 5: Post-Incident
print("\n" + "=" * 60)
print("STEP 5: Post-Incident")
print("=" * 60)

# Document incident details
incident_summary = {
    'incident_id': alert_data.get('incident_id', 'unknown'),
    'technique': 'Credential Access - Brute Force',
    'detection_time': alert_data.get('timestamp', 'unknown'),
    'affected_systems': len(affected_systems),
    'brute_force_indicators': len(suspicious_activities),
    'blocked_ips': len(set(blocked_ips + blocked_attackers)),
    'attack_vectors': list(set([vector for s in affected_systems for vector in s.get('attack_vectors', [])])),
    'containment_actions': len(isolated_hosts) + len(blocked_ips) + len(rate_limited_services),
    'eradication_actions': len(cleaned_logs) + len(blocked_attackers) + len(hardened_services),
    'recovery_actions': len(restored_services) + len(reconnected_systems) + len(validated_authentication),
    'timeline': {
        'detection': alert_data.get('timestamp', 'unknown'),
        'containment': 'completed',
        'eradication': 'completed',
        'recovery': 'completed',
        'closure': 'now'
    }
}

# Create comprehensive incident report
print(f"\n[ACTION] Creating comprehensive incident report...")
try:
    iris_report = iris.create_incident_report(
        incident_id=alert_data.get('incident_id', 'unknown'),
        title=f"Credential Access - Brute Force Incident {alert_data.get('incident_id', 'unknown')}",
        description=f"Detected and responded to brute force attacks on {len(affected_systems)} systems. {len(suspicious_activities)} suspicious indicators were identified and {len(set(blocked_ips + blocked_attackers))} attacking IPs were blocked.",
        severity='high',
        status='closed',
        details={
            'technique': 'T1110 - Brute Force',
            'indicators': [activity.get('description', 'Unknown brute force activity') for activity in suspicious_activities[:10]],
            'affected_assets': [s['hostname'] for s in affected_systems],
            'attack_vectors': incident_summary['attack_vectors'],
            'blocked_ips': list(set(blocked_ips + blocked_attackers)),
            'response_actions': {
                'detection': f"Splunk queries identified {len(suspicious_activities)} brute force indicators",
                'containment': f"Isolated {len(isolated_hosts)} hosts, blocked {len(set(blocked_ips + blocked_attackers))} IPs, rate limited {len(rate_limited_services)} services",
                'eradication': f"Cleaned logs on {len(cleaned_logs)} systems, hardened {len(hardened_services)} services, reset {len(reset_accounts) if 'reset_accounts' in locals() else 0} accounts",
                'recovery': f"Restored {len(restored_services)} services, reconnected {len(reconnected_systems)} systems, validated {len(validated_authentication)} authentications"
            },
            'threat_intelligence': {
                'misp_enrichment': len([a for a in suspicious_activities if a.get('threat_intel')]),
                'high_confidence_indicators': len([a for a in suspicious_activities if a.get('threat_intel') and len(a['threat_intel']) > 0])
            }
        }
    )
    print(f"   Incident report created in IRIS: {iris_report.get('report_id', 'unknown')}")
except Exception as e:
    print(f"   Report creation failed: {e}")

# Share indicators with threat intelligence community
print(f"\n[ACTION] Sharing indicators with threat intelligence community...")
try:
    shared_iocs = []
    for activity in suspicious_activities:
        if activity.get('threat_intel') and len(activity['threat_intel']) > 0:
            # Share attacking IPs and patterns
            ioc_attributes = []
            if activity.get('src_ip'):
                ioc_attributes.append({
                    'type': 'ip-src',
                    'value': activity['src_ip'],
                    'comment': f'Brute force attacking IP from incident {alert_data.get("incident_id", "unknown")}'
                })
            if activity.get('user_agent'):
                ioc_attributes.append({
                    'type': 'user-agent',
                    'value': activity['user_agent'],
                    'comment': f'User agent used in brute force attack'
                })
            if activity.get('attack_pattern'):
                ioc_attributes.append({
                    'type': 'text',
                    'value': activity['attack_pattern'],
                    'comment': f'Brute force attack pattern observed'
                })

            if ioc_attributes:
                misp_event = misp.create_event(
                    title=f"Brute Force Attack Detection: {activity.get('description', 'Unknown')}",
                    description=f"Brute force attack pattern detected during incident response. Attack vector: {activity.get('attack_vector', 'Unknown')}",
                    attributes=ioc_attributes
                )
                if misp_event.get('status') == 'created':
                    shared_iocs.extend([attr['value'] for attr in ioc_attributes])
                    print(f"   Shared {len(ioc_attributes)} IOCs for brute force activity")

    print(f"   Shared {len(shared_iocs)} indicators with threat intelligence community")
except Exception as e:
    print(f"   IOC sharing failed: {e}")

# Generate lessons learned
print(f"\n[ACTION] Generating lessons learned...")
lessons_learned = {
    'incident_type': 'Credential Access - Brute Force',
    'key_findings': [
        f"Brute force indicators detected: {len(suspicious_activities)} across {len(affected_systems)} systems",
        f"Primary attack vectors: {', '.join(incident_summary['attack_vectors'][:3]) if incident_summary['attack_vectors'] else 'none identified'}",
        f"IPs blocked: {len(set(blocked_ips + blocked_attackers))} unique attacking IPs",
        f"Threat intelligence enrichment: {len([a for a in suspicious_activities if a.get('threat_intel')])} activities enriched",
        f"Containment effectiveness: {len(isolated_hosts)}/{len(affected_systems)} systems isolated within detection window"
    ],
    'improvements_needed': [
        'Enhance authentication monitoring and alerting',
        'Implement real-time IP blocking and rate limiting',
        'Strengthen password policies and MFA adoption',
        'Improve geographic and behavioral access controls'
    ],
    'preventive_measures': [
        'Deploy advanced authentication protection systems',
        'Implement behavioral analytics for login anomalies',
        'Regular security policy updates for emerging brute force techniques',
        'Enhanced threat intelligence integration for known attack sources'
    ]
}

try:
    iris.add_lessons_learned(
        incident_id=alert_data.get('incident_id', 'unknown'),
        lessons=lessons_learned
    )
    print(f"   Lessons learned documented in IRIS")
except Exception as e:
    print(f"   Lessons learned documentation failed: {e}")

# Final status update
print(f"\n Incident {alert_data.get('incident_id', 'unknown')} successfully resolved:")
print(f"  - Technique: Credential Access - Brute Force (T1110)")
print(f"  - Status: Closed")
print(f"  - Systems Recovered: {len(restored_services) + len(reconnected_systems)}")
print(f"  - IPs Blocked: {len(set(blocked_ips + blocked_attackers))}")
print(f"  - Services Hardened: {len(hardened_services)}")
print(f"  - Threat Indicators Shared: {len(shared_iocs) if 'shared_iocs' in locals() else 0}")
print(f"  - Incident Report Generated: Yes")
print(f"  - Lessons Learned Documented: Yes")

print(f"\n{'=' * 60}")
print("INCIDENT RESPONSE COMPLETE")
print(f"{'=' * 60}")


## Summary

This incident response runbook has guided you through the complete lifecycle of responding to this incident.

### Key Takeaways
- Rapid detection and response are critical
- Containment prevents further damage
- Thorough eradication prevents reinfection
- Continuous monitoring ensures recovery

### Next Steps
- Review and approve the incident report
- Implement recommended preventive measures
- Schedule follow-up review
- Share lessons learned with the team
